Este conjunto de datos proviene de MIMIC, una base de datos clínica de acceso abierto muy importante en el área de la salud. El objetivo específico de este notebook es analizar los registros de ingresos hospitalarios para predecir un resultado crítico: si el paciente fallecerá o sobrevivirá durante su estancia en el hospital.
La variable objetivo principal es entrenar un modelo que pueda predecir la probabilidad de fallecimiento de un paciente en el hospital (hospital_expire_flag).
En mi fase de preparación, realicé una limpieza profunda que dividí en tres pasos clave:

Selección de variables y eliminación de ruido: Empecé aplicando lo que llamo un 'recorte de cirugía'. El dataset original tenía 19 columnas, pero decidí eliminar 16 de ellas (como row_id, subject_id y todas las fechas de ingreso o salida). Me quedé únicamente con tres variables: el tipo de admisión, la etnia y el tipo de seguro médico. Hice esto porque los IDs son solo números administrativos que no aportan valor médico, y las fechas podrían causar 'fuga de datos' (hacer trampa), ya que si el modelo ve una fecha de defunción, adivinaría el resultado sin aprender realmente los patrones de los pacientes.

Codificación numérica (Factorize): Como las tres columnas que seleccioné contienen texto (por ejemplo, 'EMERGENCY' o 'Medicare'), mi modelo no las podría procesar directamente. Para solucionar esto, utilicé la función factorize de Pandas para convertir cada categoría en un número. También generé un diccionario de mapeo para mantener el registro de qué significa cada número (por ejemplo, saber que el 0 representa una emergencia).

Normalización y Bias: Finalmente, convertí mis datos en matrices de NumPy y apliqué una Normalización Z-score. Esto es fundamental para que todas mis variables estén en la misma escala y el algoritmo de entrenamiento no se desestabilice. También añadí una columna de unos (Bias) a mi matriz de entrada para que el modelo pueda calcular correctamente el intercepto de la función logística.

In [3]:
# FASE 1: IMPORTACIONES
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot
from IPython.display import display

%matplotlib inline
print(" Fase 1: Librerías listas para el análisis de Admisiones.")

 Fase 1: Librerías listas para el análisis de Admisiones.


In [5]:
# FASE 2: EXPLORACIÓN DE DATOS EN CRUDO
ruta_archivo = '/content/sample_data/ADMISSIONS.csv'

# 1. Carga de datos
try:
    df_crudo = pd.read_csv('/Users/mac/Desktop/datasets/ MIMIC‑III /ADMISSIONS.csv')
    print(f"Archivo '{'/Users/mac/Desktop/datasets/ MIMIC‑III /ADMISSIONS.csv'}' cargado con éxito.")
except Exception as e:
    print(f"Error al cargar: {e}")

# 2. Análisis inicial de dimensiones
print(f"\n RESUMEN DE FÁBRICA:")
print(f"- Registros (Filas): {df_crudo.shape[0]}")
print(f"- Variables (Columnas): {df_crudo.shape[1]}")

# 3. Lista de columnas originales
print("\n LISTA DE COLUMNAS ORIGINALES:")
print(list(df_crudo.columns))

# 4. Vista previa de los datos
print("\nVISTA PREVIA (Top 5):")
display(df_crudo.head(5))

Archivo '/Users/mac/Desktop/datasets/ MIMIC‑III /ADMISSIONS.csv' cargado con éxito.

 RESUMEN DE FÁBRICA:
- Registros (Filas): 129
- Variables (Columnas): 19

 LISTA DE COLUMNAS ORIGINALES:
['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospital_expire_flag', 'has_chartevents_data']

VISTA PREVIA (Top 5):


,row_id,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,religion,marital_status,ethnicity,edregtime,edouttime,diagnosis,hospital_expire_flag,has_chartevents_data
0,12258,10006,142345,2164-10-23 21:09:00,2164-11-01 17:15:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,SEPARATED,BLACK/AFRICAN AMERICAN,2164-10-23 16:43:00,2164-10-23 23:00:00,SEPSIS,0,1
1,12263,10011,105331,2126-08-14 22:32:00,2126-08-28 18:59:00,2126-08-28 18:59:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Private,NaN,CATHOLIC,SINGLE,UNKNOWN/NOT SPECIFIED,NaN,NaN,HEPATITIS B,1,1
2,12265,10013,165520,2125-10-04 23:36:00,2125-10-07 15:13:00,2125-10-07 15:13:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,NaN,UNKNOWN/NOT SPECIFIED,NaN,NaN,SEPSIS,1,1
3,12269,10017,199207,2149-05-26 17:19:00,2149-06-03 18:42:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,SNF,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,2149-05-26 12:08:00,2149-05-26 19:45:00,HUMERAL FRACTURE,0,1
4,12270,10019,177759,2163-05-14 20:43:00,2163-05-15 12:00:00,2163-05-15 12:00:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,NaN,NaN,ALCOHOLIC HEPATITIS,1,1


In [6]:
# FASE 2.1: ELIMINACIÓN DE COLUMNAS (MIMIC ADMISSIONS)
# Definimos estrictamente lo que se queda
cols_a_mantener = ['admission_type', 'ethnicity', 'insurance']

# Realizamos el recorte
df = df_crudo[cols_a_mantener].copy()

print(" REPORTE DE CIRUGÍA:")
# Calculamos qué se fue para el reporte de auditoría
eliminadas = list(set(df_crudo.columns) - set(cols_a_mantener))
print(f"Columnas enviadas a la papelera: {len(eliminadas)}")
print(f"Variables que sobreviven: {list(df.columns)}")

print("\n TABLA RESULTANTE (Solo 3 columnas):")
display(df.head(5))

 REPORTE DE CIRUGÍA:
Columnas enviadas a la papelera: 16
Variables que sobreviven: ['admission_type', 'ethnicity', 'insurance']

 TABLA RESULTANTE (Solo 3 columnas):


,admission_type,ethnicity,insurance
0,EMERGENCY,BLACK/AFRICAN AMERICAN,Medicare
1,EMERGENCY,UNKNOWN/NOT SPECIFIED,Private
2,EMERGENCY,UNKNOWN/NOT SPECIFIED,Medicare
3,EMERGENCY,WHITE,Medicare
4,EMERGENCY,WHITE,Medicare


In [7]:
# FASE 2.2: ANÁLISIS DE VALORES ÚNICOS
print("AUDITORÍA DE CATEGORÍAS:")

for col in df.columns:
    print("-" * 50)
    print(f" COLUMNA: **{col}**")
    valores = df[col].unique()
    print(f"Total de categorías únicas: {len(valores)}")
    print(f"Valores: {valores}")

print("-" * 50)
print("Celda 2.2 completada. Ya conocemos el vocabulario del dataset.")

AUDITORÍA DE CATEGORÍAS:
--------------------------------------------------
 COLUMNA: **admission_type**
Total de categorías únicas: 3
Valores: <StringArray>
['EMERGENCY', 'ELECTIVE', 'URGENT']
Length: 3, dtype: str
--------------------------------------------------
 COLUMNA: **ethnicity**
Total de categorías únicas: 9
Valores: <StringArray>
[                                  'BLACK/AFRICAN AMERICAN',
                                    'UNKNOWN/NOT SPECIFIED',
                                                    'WHITE',
                                                    'OTHER',
                                                    'ASIAN',
                                       'HISPANIC OR LATINO',
                           'HISPANIC/LATINO - PUERTO RICAN',
                                         'UNABLE TO OBTAIN',
 'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE']
Length: 9, dtype: str
--------------------------------------------------
 COLUMNA: **insurance**
Total de c

In [8]:
# FASE 2.3: CODIFICACIÓN NUMÉRICA (FACTORIZE)
# 1. Recuperamos las 3 columnas de interés + el Target
cols_estudio = ['admission_type', 'ethnicity', 'insurance']
target = 'hospital_expire_flag'

# Creamos el dataframe final desde el crudo original
df = df_crudo[cols_estudio + [target]].copy()

# 2. Diccionario para guardar qué número es qué palabra (Auditoría Técnica)
mapeos = {}

for col in cols_estudio:
    # pd.factorize devuelve (los códigos, las etiquetas originales)
    codigos, etiquetas = pd.factorize(df[col])
    df[col] = codigos
    mapeos[col] = etiquetas

print("CONVERSIÓN COMPLETADA:")
for col, etiquetas in mapeos.items():
    print(f"\n Mapeo de {col}:")
    for i, nombre in enumerate(etiquetas):
        print(f"   {i} -> {nombre}")

print("\n VISTA PREVIA DEL DATASET CODIFICADO:")
display(df.head(10))

print("-" * 60)
print(f"Variables X: {cols_estudio}")
print(f"Variable Y (Target): {target}")

CONVERSIÓN COMPLETADA:

 Mapeo de admission_type:
   0 -> EMERGENCY
   1 -> ELECTIVE
   2 -> URGENT

 Mapeo de ethnicity:
   0 -> BLACK/AFRICAN AMERICAN
   1 -> UNKNOWN/NOT SPECIFIED
   2 -> WHITE
   3 -> OTHER
   4 -> ASIAN
   5 -> HISPANIC OR LATINO
   6 -> HISPANIC/LATINO - PUERTO RICAN
   7 -> UNABLE TO OBTAIN
   8 -> AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE

 Mapeo de insurance:
   0 -> Medicare
   1 -> Private
   2 -> Medicaid
   3 -> Government

 VISTA PREVIA DEL DATASET CODIFICADO:


,admission_type,ethnicity,insurance,hospital_expire_flag
0,0,0,0,0
1,0,1,1,1
2,0,1,0,1
3,0,2,0,0
4,0,2,0,1
5,0,2,0,0
6,1,2,0,0
7,0,2,0,0
8,0,2,0,0
9,0,2,0,0


------------------------------------------------------------
Variables X: ['admission_type', 'ethnicity', 'insurance']
Variable Y (Target): hospital_expire_flag


In [9]:
# FASE 3: MATEMÁTICA DE CLASIFICACIÓN (BINARY CROSS ENTROPY)
import numpy as np

def sigmoid(z):
    # Función de activación para clasificación binaria
    return 1 / (1 + np.exp(-z))

def calcularCostoBCE(theta, X, y):
    m = y.size
    h = sigmoid(X.dot(theta.T))
    eps = 1e-15 # Evita que log(0) rompa el universo (NaN)

    # FÓRMULA BINARY CROSS ENTROPY:
    # J = -1/m * sum( y*log(h) + (1-y)*log(1-h) )
    J = (1 / m) * np.sum(-y.dot(np.log(h + eps)) - (1 - y).dot(np.log(1 - h + eps)))
    return J

# --- PREPARACIÓN DE MATRICES ---
X_data = df[['admission_type', 'ethnicity', 'insurance']].values
y = df['hospital_expire_flag'].values

# Normalización Z-score (Indispensable para BCE)
mu = np.mean(X_data, axis=0)
sigma = np.std(X_data, axis=0) + 1e-8
X_norm = (X_data - mu) / sigma

# Columna de Bias (Intercepción)
X = np.concatenate([np.ones((y.size, 1)), X_norm], axis=1)

print(f"Fase 3: X preparada {X.shape}. Función BCE lista.")

Fase 3: X preparada (129, 4). Función BCE lista.
